In [2]:
# 형상 설계 제대로 되는 조건을 찾기 위한 코드
import os, glob, pathlib, json
# f = open("../.gitignore", "r")

data_dir = pathlib.Path('../tmp').resolve()
data_dirs = os.listdir(data_dir)

dicts_list = []

for computer_name in data_dirs:
    for fp in glob.glob(f'{str(data_dir)}/{computer_name}/*.json'):
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                _dict = json.load(f)
                uuid = f.name
                start = uuid.index('sim_dict_') + len('sim_dict_')
                uuid = uuid[start:]
                uuid = uuid.removesuffix(".json")
                _dict['uuid'] = uuid
                _dict['computer_name'] = computer_name
                dicts_list.append(_dict)
        except json.JSONDecodeError:
            print(f"Warning: failed to parse {fp}")

(dicts_list)

[{'aedt_dir': 'parrarel4',
  'xformer_type': 0,
  'per': 3000,
  'freq_khz': 140,
  'is_validated': True,
  'data': {},
  'o3ds': {'core_base': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D450>',
   'core_sub1': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D420>',
   'core_sub2': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D240>',
   'core_sub_g1': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D180>',
   'core_sub_g2': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D4B0>',
   'core_unite_sub_g1': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552D1E0>',
   'Tx_connect': '<ansys.aedt.core.modeler.cad.polylines.Polyline object at 0x00000296F552D120>',
   'Tx_1': '<ansys.aedt.core.modeler.cad.polylines.Polyline object at 0x00000296F552D1B0>',
   'Tx_2': '<ansys.aedt.core.modeler.cad.object_3d.Object3d object at 0x00000296F552E5F0

In [3]:
data_dict = {}

for i in dicts_list:
    dd = i['progress_dict']
    go = dd[list(dd).pop()][0]
    if not(go in data_dict.keys()):
        data_dict[go] = []
    
    data_dict[go].append(i)


In [4]:
# json.dump(data_dict, open('data_dict.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)


In [5]:
fail_level = {}
for i, k in enumerate([
"set_variable_byvalue",
"set_variable_byrange",
"validate_variable_4Tx",
"validate_variable",
"create_core",
"validate_variable_4Rx",
"create_winding_new",
"assign_mesh",
"create_excitation",
"validate_design",
"analyze_all",
"get_input_parameter",
"_get_copper_loss_parameter",
"__create_B_field",
"_get_mean_Bfield",
"coreloss_project"
]):
    fail_level[k] = i

fail_level

{'set_variable_byvalue': 0,
 'set_variable_byrange': 1,
 'validate_variable_4Tx': 2,
 'validate_variable': 3,
 'create_core': 4,
 'validate_variable_4Rx': 5,
 'create_winding_new': 6,
 'assign_mesh': 7,
 'create_excitation': 8,
 'validate_design': 9,
 'analyze_all': 10,
 'get_input_parameter': 11,
 '_get_copper_loss_parameter': 12,
 '__create_B_field': 13,
 '_get_mean_Bfield': 14,
 'coreloss_project': 15}

In [7]:
new_data_list= []

for k, v in data_dict.items():
    for dict_ in v:
        new = dict_['v']
        elapsed = int(dict_['end_time'])- int(dict_['start_time'])
        new['fail_level'] = fail_level[k]
        new['elapsed'] = elapsed
        new_data_list.append(
            new
        )

import pandas as pd
new_data_list= pd.DataFrame(new_data_list)
new_data_list.head()
new_data_list = new_data_list.drop(columns = ['seed','A', 'B', 'C', 'D', 'E'])
new_data_list.to_csv('data_분류된_형상설계.csv', index=False, encoding='utf-8')